## Import Libraries

In [2]:
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.feature import MinMaxScaler, StringIndexer, VectorAssembler
from pyspark.ml.regression import LinearRegression
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

## Start Spark Session

In [3]:
def create_spark():
    """ Create a SparkSession object. """
    spark = SparkSession.builder \
        .master("local[*]") \
        .appName("TestSuite") \
        .config(key='spark.sql.shuffle.partitions', value='4') \
        .config(key='spark.default.parallelism', value='4') \
        .config(key='spark.sql.session.timeZone', value='UTC') \
        .config(key='spark.ui.enabled', value='false') \
        .config(key='spark.app.id', value='Test') \
        .config(key='spark.driver.host', value='localhost') \
        .getOrCreate()

    return spark

In [4]:
spark = create_spark()

C:\Users\username\Projects\MachineLearning\.venv\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


## Load Data

In [5]:
path_to_data = '../../dataset/CarPrice_Assignment.csv'

car_spark_df = spark.read.csv(path_to_data, header=True, inferSchema=True)

In [6]:
features = ['symboling', 'wheelbase', 'carlength', 'carwidth', 'carheight', 'curbweight',
            'enginesize', 'boreratio', 'stroke', 'compressionratio', 'horsepower',
            'peakrpm', 'citympg', 'highwaympg']
target = 'price'

In [7]:
raw_string_columns = ['fueltype', 'aspiration', 'doornumber', 'carbody', 'drivewheel', 'enginelocation', 'enginetype', 'cylindernumber', 'fuelsystem']
indexed_string_columns = [col + "_index" for col in raw_string_columns]
encoded_string_columns = [col + "_ohe" for col in raw_string_columns]

indexer = StringIndexer(
    inputCols=raw_string_columns,
    outputCols=indexed_string_columns,
    handleInvalid="keep"  # optional
)

car_spark_df = indexer.fit(car_spark_df).transform(car_spark_df)

## Scale data

In [8]:
vectorizer = VectorAssembler(inputCols=features+indexed_string_columns, outputCol='features')
car_spark_df = vectorizer.transform(car_spark_df)
scaler = MinMaxScaler(inputCol='features', outputCol='scaled_features').fit(car_spark_df)
scaled_data = scaler.transform(car_spark_df)

In [9]:
train, test = scaled_data.randomSplit([0.7, 0.3], seed=42)

In [10]:
lr = LinearRegression(featuresCol='scaled_features', labelCol='price')
model = lr.fit(train)

In [11]:
predicted = model.transform(test)

## Calculate Error

In [14]:
predicted = predicted.withColumn('error', F.round(F.col('price') - F.col('prediction'), 2)).withColumn('error_percentage', F.round(F.col('error') / F.col('price') * 100, 2))

predicted.select('price', 'prediction', 'error', 'error_percentage').show(5)

+---------+------------------+--------+----------------+
|    price|        prediction|   error|error_percentage|
+---------+------------------+--------+----------------+
|  16500.0|16342.956255782849|  157.04|            0.95|
|  17710.0|20808.567710119285|-3098.57|           -17.5|
|  23875.0|21068.226139030787| 2806.77|           11.76|
|17859.167|23404.775429148165|-5545.61|          -31.05|
|  21105.0|17966.472680107407| 3138.53|           14.87|
+---------+------------------+--------+----------------+
only showing top 5 rows


In [15]:
avg_percentage_error = predicted.select(F.mean(F.abs(F.col('error_percentage')))).collect()[0][0]
print(f'Average percentage error: {avg_percentage_error:.2f}%')

Average percentage error: 14.64%


In [16]:
avg_error = predicted.select(F.mean(F.abs(F.col('error')))).collect()[0][0]
print(f'Average error: {avg_error:.2f}') # AKA Mean Absolute Error (MAE)

Average error: 1974.33


## Evaluation

In [13]:
metrics = {
    "r2": RegressionEvaluator(metricName="r2"),
    "rmse": RegressionEvaluator(metricName="rmse"),
    "mae": RegressionEvaluator(metricName="mae"),
}

for name, evaluator in metrics.items():
    evaluator.setLabelCol(target)
    print(f'Metric: {name} =  {evaluator.evaluate(predicted)}')
# Metric: r2 =  0.878333540078323
# Metric: rmse =  2814.7716827826303
# Metric: mae =  1974.33215551965

Metric: r2 =  0.8783335400783229
Metric: rmse =  2814.7716827826307
Metric: mae =  1974.3321555196508
